# Credit Card Fraud Detection on Imbalanced Data Using Machine Learning

---

## Table of Contents
1. [Introduction](#introduction)
2. [Dataset Overview](#dataset-overview)
3. [Exploratory Data Analysis (EDA)](#eda)
4. [Handling Imbalanced Data](#handling-imbalanced-data)
5. [Modeling and Evaluation](#modeling-and-evaluation)
6. [Save the Trained Models](#saving-model)
7. [Conclusion](#conclusion)

---

## Introduction <a name="introduction"></a>

Credit card fraud detection is a crucial application of machine learning in financial systems. However, fraud cases are rare, making the dataset highly imbalanced, which poses challenges for predictive modeling. This notebook demonstrates how to effectively address these challenges using machine learning techniques.

**Objectives:**
- Detect fraudulent transactions from credit card data.
- Handle the issue of data imbalance.
- Compare model performance and select the best approach.

---

## Dataset Overview <a name="dataset-overview"></a>

The dataset used for this project contains transactions made by credit cards. It includes a highly imbalanced set of observations, where fraudulent transactions are a small fraction of the total.

**Features:**
- **V1-V28**: Principal components obtained via PCA.
- **Amount**: Transaction amount.
- **Time**: Time elapsed from the first transaction.
- **Class**: Fraud status (0 for non-fraud, 1 for fraud).

---

## Exploratory Data Analysis (EDA) <a name="eda"></a>

To understand the data distribution and identify potential issues:

- Plot the distribution of features.
- Visualize the imbalance in the `Class` variable.
- Investigate correlations between features.

---

## Handling Imbalanced Data <a name="handling-imbalanced-data"></a>

Given the highly imbalanced nature of the dataset, special techniques are applied:

- **Resampling**:
  - **Oversampling**: Synthetic Minority Oversampling Technique (SMOTE).
  - **Undersampling**: Random undersampling.
  
- **Evaluation Metrics**:
  - Accuracy may not be suitable for imbalanced data. We use:
    - Precision, Recall, F1-score, and ROC-AUC.

---

## Modeling and Evaluation <a name="modeling-and-evaluation"></a>

We apply and evaluate several machine learning models:

1. **Logistic Regression**
2. **Decision Tree**
3. **Random Forest**

For each model:
- Train on the imbalanced dataset and again on resampled dataset.
- Evaluate performance using confusion matrices.
- Compare results based on evaluation metrics.

---

## Save the Trained Models <a name="saving-model"></a>

After training and evaluating the machine learning models, it is crucial to save them for future use. This allows you to avoid retraining the models every time you need to make predictions and facilitates easy deployment.

In this step, we will save the trained models using Python's `joblib` library. This will enable us to load and use the models later without needing to retrain them.

We will save the model for `RandomForestClassifier`


**Let’s proceed with coding...**

### Import Required Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from imblearn.over_sampling import SMOTE

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score


from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

import joblib

ModuleNotFoundError: No module named 'pandas'

### Load and Explore the Dataset

In [3]:
!pip install imbalanced-learn

  Using cached imbalanced_learn-0.14.1-py3-none-any.whl.metadata (8.9 kB)
  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached sklearn_compat-0.1.5-py3-none-any.whl.metadata (20 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached imbalanced_learn-0.14.1-py3-none-any.whl (235 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   - -------------------------------------- 0.3

In [4]:

df = pd.read_csv('creditcard.csv')

NameError: name 'pd' is not defined

In [ ]:

df = pd.read_csv('creditcard.csv')

In [ ]:
# Display first few rows
df.head()

In [ ]:
# Display All Columns
pd.options.display.max_columns = None
df.head()

In [ ]:
# Display dataset shape
df.shape

In [ ]:
print("Number of columns: {}".format(df.shape[1]))
print("Number of rows: {}".format(df.shape[0]))

In [ ]:
# Display dataset information
df.info()

In [ ]:
# Summary statistics
df.describe()

### Visualize Class Imbalance
Use seaborn to visualize the distribution of the target variable to understand class imbalance.

In [ ]:
df['Class'].value_counts()

In [ ]:
# Visualize the distribution of the target variable
plt.style.use('ggplot')
plt.figure(figsize=(8, 6))
colors = [ "#DF0101", "#0101DF"]
sns.countplot(x='Class', data = df, palette=colors, hue='Class')
plt.title('Distribution of Fraud vs Normal Transactions \n (0: Normal || 1: Fraud)', fontsize=11)
plt.xlabel('Class')
plt.ylabel('Count')
plt.show()

### Data Preprocessing
- Check for missing or null values.
- Handle duplicates.
- Scale the features (`Amount`, `Time`).
- Split features and target.


In [ ]:
# Check for missing values
df.isnull().sum().sum()

In [ ]:
# Check for duplicate rows
df.duplicated().sum()

In [ ]:
# Drop duplicate rows
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

In [ ]:
# Standardize 'Amount' and 'Time'
scaler = StandardScaler()
df['Amount'] = scaler.fit_transform(df[['Amount']])
df['Time'] = scaler.fit_transform(df[['Time']])

In [ ]:
df.head()

In [ ]:
# Split features and target
X = df.drop('Class', axis=1)
y = df['Class']

### Step 7: Investigate Correlations Between Features

Correlation analysis helps to understand how features in the dataset relate to each other. High correlation between features might indicate redundancy, while low correlation can suggest that features capture different aspects of the data.

Let's calculate and visualize the correlation matrix.


In [ ]:
# Calculate the correlation matrix
corr_matrix = df.corr()
corr_matrix

In [ ]:
# Plot the Heatmap
plt.figure(figsize=(15,12))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', linewidths=0.5, annot_kws={"size":7})
plt.title('Correlation Matrix of Features')
plt.show()

### Split Data into Training and Test Sets

In [ ]:
# Train-test split with imbalanced data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Check the shape
X_train.shape

In [ ]:
X_test.shape

### Train and Evaluate Model on Imbalanced Data
Train a model and evaluate its performance on the imbalanced dataset.

In [ ]:
def train_model(X_train, X_test, y_train, y_test):
    """
    Trains and evaluates multiple classifiers on the given training and test datasets.

    This function takes training and test feature sets and labels, trains three different classifiers
    (Logistic Regression, Decision Tree Classifier, and RandomForestClassifier) on the training data, 
    and evaluates their performance on the test data. For each classifier, it prints the confusion matrix, 
    classification report, ROC-AUC score, and plots the ROC curve.

    Parameters:
    X_train (pd.DataFrame or np.ndarray): Features of the training data.
    X_test (pd.DataFrame or np.ndarray): Features of the test data.
    y_train (pd.Series or np.ndarray): Labels of the training data.
    y_test (pd.Series or np.ndarray): Labels of the test data.

    Returns:
    None: This function does not return any values but prints evaluation metrics and plots.

    Notes:
    - The function assumes that the test data includes both positive and negative class samples.
    - ROC-AUC scores and ROC curves are only meaningful if the classifier provides probability estimates 
      for the positive class.
    - This function will display the ROC curves in separate plots for each classifier.
    """
    
    classifier = {
        "Logistic Regression": LogisticRegression(),
        "Decision Tree Classifier": DecisionTreeClassifier(),
        "RandomForestClassifier": RandomForestClassifier(random_state=42)
    }

    for name, model in classifier.items():
        print(f"\n================ {name} ================\n")
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(X_test)
        
        # Confusion matrix
        conf_matrix = confusion_matrix(y_test, y_pred)
        print(f"Confusion Matrix:\n{conf_matrix}\n")

        # Accuracy
        print(f"\nAccuracy: {accuracy_score(y_test, y_pred)}\n")
        
        # Classification report
        clf_report = classification_report(y_test, y_pred)
        print(f"\nClassification Report:\n{clf_report}\n")
    
        # ROC-AUC Score
        roc_auc = roc_auc_score(y_test, y_pred)
        print(f"ROC-AUC Score (Imbalanced Data): {roc_auc}\n")
    
        # Plot ROC Curve
        fpr, tpr, thresholds = roc_curve(y_test, model.predict_proba(X_test)[:, 1])
        plt.plot(fpr, tpr, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC Curve (Imbalanced Data)\nusing {name}', fontsize=10)
        plt.legend(loc='best')
        plt.show()
        print("\n")


In [ ]:
# Call the train_model function
train_model(X_train, X_test, y_train, y_test)

### Handling Imbalanced Data
Two techniques:
- Undersampling
- Oversampling

#### Undersampling

In [ ]:
# Separate normal and fraud transactions
normal = df[df['Class']==0]
fraud = df[df['Class']==1]

# Print shape for reference
print(f"Normal transactions shape: {normal.shape}")
print(f"Fraud transactions shape: {fraud.shape}")

In [ ]:
# Undersample normal transactions
normal_sample = normal.sample(n=fraud.shape[0])

In [ ]:
# Print the shape of the new normal transactions
print(f"New normal transactions shape: {normal_sample.shape}")

In [ ]:
# Concate updated normal transcations with old fraud transaction and make a new df
new_df = pd.concat([normal_sample, fraud], ignore_index=True)

In [ ]:
# Print few rows of new df
new_df.head()

In [ ]:
# Check new class distribution
new_df['Class'].value_counts()

In [ ]:
# Split new df into X and y
X = new_df.drop('Class', axis=1)
y = new_df['Class']

In [ ]:
# Train test split on Undersampled data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Train models with undersampled data
# Call the train_model function
train_model(X_train, X_test, y_train, y_test)

#### Oversampling
- Use `SMOTE` to balance the dataset by oversampling the minority class

In [ ]:
# Split features and target
X = df.drop('Class', axis=1)
y = df['Class']

In [ ]:
# Apply SMOTE for oversampling
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

In [ ]:
# Check new class distribution
y_resampled.value_counts()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

In [ ]:
# Train models with undersampled data
# Call the train_model function
train_model(X_train, X_test, y_train, y_test)

### Save the Trained Model (Decision Tree Classifier)

In [ ]:
# Pick a model, e.g. Decision Tree Classifier
# Fit the model with X_train, y_train
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

In [ ]:
# Save the model
joblib.dump(model, "credit_card_model.pkl")

### Load the Model

In [ ]:
# Load the model
model = joblib.load("credit_card_model.pkl")

In [ ]:
# Predict with a sample data
pred = model.predict([[-1.2063166480452974,-0.653464067093327,1.15579454161356,1.4398458100309,-0.0483979577939286,-0.257954764175468,-0.763320426103366,0.339229688923037,-0.768705965846787,-0.115541693321453,-0.20021646873148,-0.650926246487322,-0.735778340137806,-1.3940656101548,0.447721760826057,0.98477163074674,0.271223077633162,-0.251055900420813,-0.165413052650476,0.0091942158033362,-0.160498072645411,0.518041029678345,-0.970619090556498,0.104889604203672,0.307935462300307,-0.222502578722938,0.0825004649897294,0.291624326333603,0.125488524044667,-0.3442213776454372]])

# Print the prediction
if pred == 0:
    print("Normal Transaction")
else:
    print("Fraud Transaction")

## Conclusion <a name="conclusion"></a>

In this analysis, we performed a comprehensive exploration and modeling process for credit card fraud detection using an imbalanced dataset. The steps included:

1. **Data Exploration and Preprocessing**: We began by loading and exploring the dataset, visualizing class imbalance, and performing necessary preprocessing steps.

2. **Correlation Analysis**: By investigating the correlations between features, we gained insights into how features interact with each other. This helped in identifying potential redundancies and understanding feature relationships.

3. **Handling Imbalanced Data**: We applied techniques such as undersampling and oversampling to address the class imbalance. This was crucial in ensuring that our models could better learn from the minority class.

4. **Model Training and Evaluation**: We trained and evaluated various classifiers, including Logistic Regression, Decision Tree, and RandomForest. We assessed model performance using metrics such as confusion matrices, classification reports, ROC-AUC scores, and ROC curves.

5. **Model Saving**: The trained models were saved for future use, ensuring that we can easily load and apply them for predictions on new data.

**Key Findings**:

- **Feature Correlations**: Our correlation analysis revealed important relationships between features. This understanding can guide feature selection and engineering in future analyses.

- **Model Performance**: The RandomForestClassifier demonstrated high accuracy in detecting fraud, showing that it is a strong candidate for deployment. The ROC-AUC scores and ROC curves provided insights into each model’s performance, particularly in distinguishing between fraudulent and non-fraudulent transactions.

- **Impact of Imbalance Handling**: Techniques for balancing the dataset were essential in improving model performance and ensuring that the minority class (fraudulent transactions) was adequately represented in the training process.

Overall, this analysis has provided a robust framework for credit card fraud detection. The insights gained from feature correlations and model evaluations will be instrumental in refining our approach and improving detection capabilities. Future work could involve fine-tuning models further, experimenting with additional features, and exploring other advanced techniques for handling imbalanced data.
